In [2]:
from pathlib import Path

import polars as pl

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent.resolve()

OBSERVATIONS_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "observations"
    / "master_observations.parquet"
)

BASELINE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "baselines"
    / "precipitation_amount.parquet"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "anomalies"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_FILE = (
    OUTPUT_DIR
    / "precipitation_amount.parquet"
)

print("=" * 60)
print("COMPUTE RAINFALL ANOMALIES")
print("=" * 60)

assert OBSERVATIONS_FILE.exists()
assert BASELINE_FILE.exists()

print(f"Observations : {OBSERVATIONS_FILE}")
print(f"Baseline     : {BASELINE_FILE}")
print(f"Output       : {OUTPUT_FILE}")
print("=" * 60)

COMPUTE RAINFALL ANOMALIES
Observations : C:\Projects\ews\zoryn\data\processed\observations\master_observations.parquet
Baseline     : C:\Projects\ews\zoryn\data\processed\baselines\precipitation_amount.parquet
Output       : C:\Projects\ews\zoryn\data\processed\anomalies\precipitation_amount.parquet


In [3]:
# ============================================================
# LOAD DATA
# ============================================================

observations = (
    pl.scan_parquet(OBSERVATIONS_FILE)
    .with_columns(
        pl.col("timestamp")
        .str.slice(5, 2)
        .cast(pl.UInt8)
        .alias("month")
    )
)

baseline = pl.scan_parquet(BASELINE_FILE)

obs_summary = (
    observations
    .select(
        [
            pl.len().alias("rows"),
            pl.n_unique("h3_id").alias("h3_cells"),
            pl.n_unique("month").alias("months"),
        ]
    )
    .collect()
)

baseline_summary = (
    baseline
    .select(
        [
            pl.len().alias("rows"),
            pl.n_unique("h3_id").alias("h3_cells"),
            pl.n_unique("month").alias("months"),
        ]
    )
    .collect()
)

print("=" * 60)
print("INPUT VALIDATION")
print("=" * 60)
print("Observations")
print(obs_summary)
print()
print("Baseline")
print(baseline_summary)
print("=" * 60)

assert obs_summary["h3_cells"][0] == baseline_summary["h3_cells"][0]
assert baseline_summary["months"][0] == 12

INPUT VALIDATION
Observations
shape: (1, 3)
┌──────────┬──────────┬────────┐
│ rows     ┆ h3_cells ┆ months │
│ ---      ┆ ---      ┆ ---    │
│ u32      ┆ u32      ┆ u32    │
╞══════════╪══════════╪════════╡
│ 11679216 ┆ 42316    ┆ 12     │
└──────────┴──────────┴────────┘

Baseline
shape: (1, 3)
┌────────┬──────────┬────────┐
│ rows   ┆ h3_cells ┆ months │
│ ---    ┆ ---      ┆ ---    │
│ u32    ┆ u32      ┆ u32    │
╞════════╪══════════╪════════╡
│ 507792 ┆ 42316    ┆ 12     │
└────────┴──────────┴────────┘


In [4]:
# ============================================================
# JOIN OBSERVATIONS WITH CLIMATOLOGY
# ============================================================

anomalies = (
    observations
    .join(
        baseline,
        on=[
            "h3_id",
            "month",
        ],
        how="left",
    )
)

summary = (
    anomalies
    .select(
        [
            pl.len().alias("rows"),
            pl.col("mean").is_null().sum().alias("missing_mean"),
            pl.col("std").is_null().sum().alias("missing_std"),
        ]
    )
    .collect()
)

print("=" * 60)
print("JOIN VALIDATION")
print("=" * 60)
print(summary)
print("=" * 60)

assert summary["missing_mean"][0] == 0
assert summary["missing_std"][0] == 0

JOIN VALIDATION
shape: (1, 3)
┌──────────┬──────────────┬─────────────┐
│ rows     ┆ missing_mean ┆ missing_std │
│ ---      ┆ ---          ┆ ---         │
│ u32      ┆ u32          ┆ u32         │
╞══════════╪══════════════╪═════════════╡
│ 11679216 ┆ 0            ┆ 0           │
└──────────┴──────────────┴─────────────┘


In [5]:
# ============================================================
# COMPUTE ANOMALIES
# ============================================================

EPSILON = 1e-6

anomalies = (
    anomalies
    .with_columns(
        [
            (pl.col("value") - pl.col("mean"))
            .alias("anomaly"),

            pl.when(pl.col("std") > EPSILON)
            .then(
                (pl.col("value") - pl.col("mean"))
                / pl.col("std")
            )
            .otherwise(0.0)
            .alias("z_score"),
        ]
    )
)

summary = (
    anomalies
    .select(
        [
            pl.col("anomaly").min().alias("min_anomaly"),
            pl.col("anomaly").max().alias("max_anomaly"),
            pl.col("z_score").min().alias("min_z"),
            pl.col("z_score").max().alias("max_z"),
        ]
    )
    .collect()
)

print("=" * 60)
print("ANOMALY SUMMARY")
print("=" * 60)
print(summary)
print("=" * 60)

ANOMALY SUMMARY
shape: (1, 4)
┌─────────────┬─────────────┬───────────┬──────────┐
│ min_anomaly ┆ max_anomaly ┆ min_z     ┆ max_z    │
│ ---         ┆ ---         ┆ ---       ┆ ---      │
│ f32         ┆ f32         ┆ f32       ┆ f32      │
╞═════════════╪═════════════╪═══════════╪══════════╡
│ -186.139374 ┆ 418.70636   ┆ -2.204687 ┆ 4.252658 │
└─────────────┴─────────────┴───────────┴──────────┘


In [7]:
# ============================================================
# ANOMALY QA
# ============================================================

summary = (
    anomalies
    .select(
        [
            pl.len().alias("rows"),
            pl.col("anomaly").is_null().sum().alias("null_anomaly"),
            pl.col("z_score").is_null().sum().alias("null_z_score"),
            pl.col("h3_id").n_unique().alias("h3_cells"),
            pl.col("timestamp").n_unique().alias("timestamps"),
        ]
    )
    .collect()
)

expected_rows = (
    summary["h3_cells"][0]
    * summary["timestamps"][0]
)

print("=" * 60)
print("ANOMALY QA")
print("=" * 60)
print(summary)
print("=" * 60)

assert summary["rows"][0] == expected_rows
assert summary["null_anomaly"][0] == 0
assert summary["null_z_score"][0] == 0

ANOMALY QA
shape: (1, 5)
┌──────────┬──────────────┬──────────────┬──────────┬────────────┐
│ rows     ┆ null_anomaly ┆ null_z_score ┆ h3_cells ┆ timestamps │
│ ---      ┆ ---          ┆ ---          ┆ ---      ┆ ---        │
│ u32      ┆ u32          ┆ u32          ┆ u32      ┆ u32        │
╞══════════╪══════════════╪══════════════╪══════════╪════════════╡
│ 11679216 ┆ 0            ┆ 0            ┆ 42316    ┆ 276        │
└──────────┴──────────────┴──────────────┴──────────┴────────────┘


In [8]:
# ============================================================
# SAVE ANOMALIES
# ============================================================

(
    anomalies
    .drop("month")
    .collect()
    .write_parquet(
        OUTPUT_FILE,
        compression="zstd",
    )
)

print("=" * 60)
print("ANOMALIES SAVED")
print("=" * 60)
print(f"File : {OUTPUT_FILE}")
print(f"Rows : 11,680,044")
print("=" * 60)

ANOMALIES SAVED
File : C:\Projects\ews\zoryn\data\processed\anomalies\precipitation_amount.parquet
Rows : 11,680,044
